# Notebook 05 — Comparação didática das quatro lógicas

Neste notebook, vamos comparar as quatro lógicas estudadas:

- lógica multivalorada;
- lógica fuzzy;
- lógica paracompleta;
- lógica paraconsistente.

O foco não é dizer qual é “a melhor” em termos absolutos.  
O foco é observar **como cada lógica se comporta** diante de três situações:

1. base original;
2. base com lacunas;
3. fontes contraditórias.

Cada lógica responde melhor a um tipo de problema.

## Etapa 1 — Enviar o arquivo da base

In [ ]:
# ============================================================
# 1. Faça upload do arquivo diabetes_data_upload.csv
# ============================================================

from google.colab import files
import io
import pandas as pd

print("Selecione o arquivo diabetes_data_upload.csv no seu computador.")
uploaded = files.upload()

# Pega automaticamente o primeiro arquivo enviado
nome_arquivo = list(uploaded.keys())[0]

df = pd.read_csv(io.BytesIO(uploaded[nome_arquivo]))

print("Arquivo carregado com sucesso!")
print(f"Nome do arquivo: {nome_arquivo}")
print(f"Quantidade de linhas: {df.shape[0]}")
print(f"Quantidade de colunas: {df.shape[1]}")

df.head()

## Etapa 2 — Preparar funções auxiliares

In [ ]:
# ============================================================
# Bibliotecas e funções auxiliares
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Coluna-alvo da base
TARGET = "class"

# Marcadores clínico-sintomáticos presentes na base
SINTOMAS = [
    "Polyuria",
    "Polydipsia",
    "sudden weight loss",
    "weakness",
    "Polyphagia",
    "Genital thrush",
    "visual blurring",
    "Itching",
    "Irritability",
    "delayed healing",
    "partial paresis",
    "muscle stiffness",
    "Alopecia",
    "Obesity",
]

# Sintomas escolhidos como centrais para simular lacunas ou conflito
# A escolha é didática: são sintomas que aparecem com relevância clínica
# e ajudam a mostrar o comportamento das lógicas.
SINTOMAS_CENTRAIS = [
    "Polyuria",
    "Polydipsia",
    "sudden weight loss",
    "weakness",
    "Polyphagia",
    "partial paresis",
]

# Pesos heurísticos.
# Eles não foram aprendidos por machine learning.
# São apenas uma forma didática de dizer:
# "alguns sintomas pesam mais na suspeita do que outros".
PESOS = {
    "Polyuria": 3.0,
    "Polydipsia": 3.0,
    "sudden weight loss": 2.0,
    "weakness": 1.0,
    "Polyphagia": 1.5,
    "Genital thrush": 0.5,
    "visual blurring": 1.0,
    "Itching": 0.25,
    "Irritability": 1.0,
    "delayed healing": 0.75,
    "partial paresis": 2.0,
    "muscle stiffness": 0.5,
    "Alopecia": 0.25,
    "Obesity": 0.25,
}

def sim_nao_para_numero(valor):
    '''
    Converte respostas categóricas da base para números.

    Yes -> 1
    No  -> 0
    NaN -> NaN

    Essa conversão permite que a lógica opere sobre evidências.
    '''
    if pd.isna(valor):
        return np.nan

    valor = str(valor).strip().lower()

    if valor == "yes":
        return 1.0
    elif valor == "no":
        return 0.0
    else:
        return np.nan

def pertinencia_idade(idade):
    '''
    Transforma idade em uma evidência gradual simples.

    Menos de 30 anos: pertinência 0
    Entre 30 e 60 anos: crescimento linear
    Acima de 60 anos: pertinência 1

    Não é uma regra médica definitiva.
    É apenas uma simplificação didática para mostrar como
    variáveis numéricas podem entrar no raciocínio lógico.
    '''
    if pd.isna(idade):
        return 0.5

    if idade < 30:
        return 0.0
    elif idade > 60:
        return 1.0
    else:
        return (idade - 30) / 30

def classe_para_binario(valor):
    '''
    Converte a classe original da base para comparação simples.

    Positive -> 1
    Negative -> 0
    '''
    return 1 if valor == "Positive" else 0

def mostrar_distribuicao_decisoes(resultado, coluna="decisao", titulo="Distribuição das decisões"):
    '''
    Plota um gráfico simples com a distribuição das saídas produzidas pela lógica.
    '''
    contagem = resultado[coluna].value_counts()

    plt.figure(figsize=(8, 4))
    barras = plt.bar(contagem.index.astype(str), contagem.values)
    plt.title(titulo)
    plt.xlabel("Saída produzida pela lógica")
    plt.ylabel("Quantidade de registros")
    plt.xticks(rotation=25, ha="right")

    for barra in barras:
        altura = barra.get_height()
        plt.text(
            barra.get_x() + barra.get_width()/2,
            altura,
            str(int(altura)),
            ha="center",
            va="bottom"
        )

    plt.tight_layout()
    plt.show()

def resumo_simples(resultado, coluna_decisao="decisao"):
    '''
    Gera um resumo didático da saída da lógica.

    A ideia não é tratar a lógica como um classificador tradicional,
    mas observar como ela decide, quando se abstém e como distribui
    suas respostas.
    '''
    total = len(resultado)

    resumo = resultado[coluna_decisao].value_counts().reset_index()
    resumo.columns = ["decisao", "quantidade"]
    resumo["percentual"] = 100 * resumo["quantidade"] / total

    return resumo

## Etapa 3 — Conhecer a base

In [ ]:
# ============================================================
# Olhando a base antes de aplicar qualquer lógica
# ============================================================

print("Dimensão da base:", df.shape)
print("\nColunas disponíveis:")
print(df.columns.tolist())

print("\nDistribuição da classe:")
display(df[TARGET].value_counts().reset_index().rename(columns={"index": "classe", TARGET: "quantidade"}))

print("\nQuantidade de valores ausentes por coluna:")
display(df.isna().sum().reset_index().rename(columns={"index": "coluna", 0: "ausentes"}))

print("\nResumo da idade:")
display(df["Age"].describe())

## Etapa 4 — Criar cenários de lacuna e conflito

In [ ]:
# ============================================================
# Criando uma cópia da base com lacunas simuladas
# ============================================================

def criar_base_incompleta(base, taxa_lacunas=0.20, semente=42):
    '''
    Cria uma versão da base com alguns sintomas centrais apagados.

    Isso simula uma situação comum em saúde:
    nem toda informação foi registrada, perguntada ou confirmada.
    '''
    base_incompleta = base.copy()
    rng = np.random.default_rng(semente)

    for coluna in SINTOMAS_CENTRAIS:
        mascara = rng.random(len(base_incompleta)) < taxa_lacunas
        base_incompleta.loc[mascara, coluna] = np.nan

    return base_incompleta

df_incompleta = criar_base_incompleta(df, taxa_lacunas=0.20)

print("Base original:")
display(df[SINTOMAS_CENTRAIS].head())

print("Base com lacunas simuladas:")
display(df_incompleta[SINTOMAS_CENTRAIS].head())

print("Valores ausentes após simulação:")
display(df_incompleta[SINTOMAS_CENTRAIS].isna().sum())

In [ ]:
# ============================================================
# Criando duas fontes contraditórias
# ============================================================

def criar_fontes_contraditorias(base, taxa_conflito=0.20, semente=42):
    '''
    Cria duas versões da base:

    Fonte A: mantém os dados originais.
    Fonte B: inverte parte dos sintomas centrais.

    Exemplo:
    Se a fonte A diz Polyuria = Yes,
    a fonte B pode dizer Polyuria = No.

    Isso simula conflito entre fontes, prontuários, entrevistas
    ou sistemas diferentes.
    '''
    fonte_a = base.copy()
    fonte_b = base.copy()

    rng = np.random.default_rng(semente)

    for coluna in SINTOMAS_CENTRAIS:
        mascara = rng.random(len(fonte_b)) < taxa_conflito
        fonte_b.loc[mascara, coluna] = fonte_b.loc[mascara, coluna].map({
            "Yes": "No",
            "No": "Yes"
        })

    return fonte_a, fonte_b

fonte_a, fonte_b = criar_fontes_contraditorias(df, taxa_conflito=0.20)

print("Fonte A:")
display(fonte_a[SINTOMAS_CENTRAIS].head())

print("Fonte B:")
display(fonte_b[SINTOMAS_CENTRAIS].head())

## Etapa 5 — Implementar as quatro lógicas

Nesta célula, concentramos as quatro implementações.

Elas são versões didáticas, com regras simples e transparentes, para mostrar a diferença entre os formalismos.

In [ ]:
# ------------------------------------------------------------
# 1. Lógica Multivalorada
# ------------------------------------------------------------

def decisao_multivalorada(linha):
    valores = []
    pesos = []

    for sintoma in SINTOMAS:
        valor = sim_nao_para_numero(linha[sintoma])
        if pd.isna(valor):
            valor = 0.5
        valores.append(valor)
        pesos.append(PESOS[sintoma])

    valores.append(pertinencia_idade(linha["Age"]))
    pesos.append(0.75)

    escore = np.average(valores, weights=pesos)
    lacunas_centrais = sum(pd.isna(linha[coluna]) for coluna in SINTOMAS_CENTRAIS)

    if lacunas_centrais >= 3:
        decisao = "indeterminado"
    elif escore >= 0.60:
        decisao = "alto"
    elif escore >= 0.35:
        decisao = "moderado"
    else:
        decisao = "baixo"

    return pd.Series({
        "escore": escore,
        "lacunas_centrais": lacunas_centrais,
        "decisao": decisao
    })

def aplicar_multivalorada(base):
    resultado = base.copy()
    saidas = resultado.apply(decisao_multivalorada, axis=1)
    return pd.concat([resultado, saidas], axis=1)


# ------------------------------------------------------------
# 2. Lógica Fuzzy
# ------------------------------------------------------------

def decisao_fuzzy(linha):
    soma_ponderada = 0
    soma_pesos = 0

    for sintoma in SINTOMAS:
        valor = sim_nao_para_numero(linha[sintoma])
        if pd.isna(valor):
            valor = 0.5

        soma_ponderada += valor * PESOS[sintoma]
        soma_pesos += PESOS[sintoma]

    soma_ponderada += pertinencia_idade(linha["Age"]) * 0.75
    soma_pesos += 0.75

    grau_risco = soma_ponderada / soma_pesos

    if grau_risco >= 0.65:
        decisao = "alta_suspeita"
    elif grau_risco >= 0.35:
        decisao = "suspeita_moderada"
    else:
        decisao = "baixa_suspeita"

    return pd.Series({
        "escore": grau_risco,
        "lacunas_centrais": sum(pd.isna(linha[coluna]) for coluna in SINTOMAS_CENTRAIS),
        "decisao": decisao
    })

def aplicar_fuzzy(base):
    resultado = base.copy()
    saidas = resultado.apply(decisao_fuzzy, axis=1)
    return pd.concat([resultado, saidas], axis=1)


# ------------------------------------------------------------
# 3. Lógica Paracompleta
# ------------------------------------------------------------

def decisao_paracompleta(linha):
    lacunas_centrais = sum(pd.isna(linha[coluna]) for coluna in SINTOMAS_CENTRAIS)

    valores = []
    pesos = []

    for sintoma in SINTOMAS:
        valor = sim_nao_para_numero(linha[sintoma])
        if not pd.isna(valor):
            valores.append(valor)
            pesos.append(PESOS[sintoma])

    if len(valores) == 0:
        return pd.Series({
            "escore": np.nan,
            "lacunas_centrais": lacunas_centrais,
            "decisao": "inconclusivo"
        })

    valores.append(pertinencia_idade(linha["Age"]))
    pesos.append(0.75)

    escore = np.average(valores, weights=pesos)

    if lacunas_centrais >= 2:
        decisao = "inconclusivo"
    elif lacunas_centrais == 1 and 0.35 <= escore < 0.65:
        decisao = "suspeita_com_evidencia_incompleta"
    elif escore >= 0.65:
        decisao = "suspeita_alta"
    elif escore >= 0.35:
        decisao = "suspeita_moderada"
    else:
        decisao = "suspeita_baixa"

    return pd.Series({
        "escore": escore,
        "lacunas_centrais": lacunas_centrais,
        "decisao": decisao
    })

def aplicar_paracompleta(base):
    resultado = base.copy()
    saidas = resultado.apply(decisao_paracompleta, axis=1)
    return pd.concat([resultado, saidas], axis=1)


# ------------------------------------------------------------
# 4. Lógica Paraconsistente
# ------------------------------------------------------------

def decisao_paraconsistente(linha_a, linha_b=None):
    if linha_b is None:
        linha_b = linha_a

    evidencia_favoravel = 0
    evidencia_contraria = 0
    soma_pesos = 0

    conflitos = 0
    conflitos_centrais = 0

    for sintoma in SINTOMAS:
        valor_a = sim_nao_para_numero(linha_a[sintoma])
        valor_b = sim_nao_para_numero(linha_b[sintoma])
        peso = PESOS[sintoma]

        if pd.isna(valor_a) and pd.isna(valor_b):
            continue

        valores = [valor for valor in [valor_a, valor_b] if not pd.isna(valor)]

        if len(valores) == 0:
            continue

        media_favoravel = np.mean(valores)
        media_contraria = 1 - media_favoravel

        evidencia_favoravel += media_favoravel * peso
        evidencia_contraria += media_contraria * peso
        soma_pesos += peso

        if len(valores) == 2 and valor_a != valor_b:
            conflitos += 1
            if sintoma in SINTOMAS_CENTRAIS:
                conflitos_centrais += 1

    idade_fav = pertinencia_idade(linha_a["Age"])
    evidencia_favoravel += idade_fav * 0.75
    evidencia_contraria += (1 - idade_fav) * 0.75
    soma_pesos += 0.75

    grau_favoravel = evidencia_favoravel / soma_pesos
    grau_contrario = evidencia_contraria / soma_pesos
    grau_certeza = grau_favoravel - grau_contrario

    if conflitos_centrais >= 1 or conflitos >= 3:
        decisao = "inconsistente"
    elif grau_certeza >= 0.30:
        decisao = "alto"
    elif grau_certeza <= -0.30:
        decisao = "baixo"
    else:
        decisao = "indeterminado"

    return pd.Series({
        "escore": grau_certeza,
        "conflitos": conflitos,
        "conflitos_centrais": conflitos_centrais,
        "decisao": decisao
    })

def aplicar_paraconsistente(fonte_a, fonte_b=None):
    resultado = fonte_a.copy()

    if fonte_b is None:
        saidas = fonte_a.apply(lambda linha: decisao_paraconsistente(linha), axis=1)
    else:
        saidas = pd.DataFrame([
            decisao_paraconsistente(fonte_a.iloc[i], fonte_b.iloc[i])
            for i in range(len(fonte_a))
        ])

    return pd.concat(
        [resultado.reset_index(drop=True), saidas.reset_index(drop=True)],
        axis=1
    )

## Etapa 6 — Aplicar as lógicas nos cenários

Agora cada lógica será aplicada no cenário em que sua característica fica mais evidente.

- Multivalorada: base original e base com lacunas.
- Fuzzy: base original e base com lacunas.
- Paracompleta: base original e base com lacunas.
- Paraconsistente: base original e fontes contraditórias.

In [ ]:
resultados = {}

resultados["Multivalorada — original"] = aplicar_multivalorada(df)
resultados["Multivalorada — lacunas"] = aplicar_multivalorada(df_incompleta)

resultados["Fuzzy — original"] = aplicar_fuzzy(df)
resultados["Fuzzy — lacunas"] = aplicar_fuzzy(df_incompleta)

resultados["Paracompleta — original"] = aplicar_paracompleta(df)
resultados["Paracompleta — lacunas"] = aplicar_paracompleta(df_incompleta)

resultados["Paraconsistente — sem conflito"] = aplicar_paraconsistente(df)
resultados["Paraconsistente — com conflito"] = aplicar_paraconsistente(fonte_a, fonte_b)

print("Execução concluída.")

## Etapa 7 — Comparar as saídas

A comparação abaixo mostra quantos registros caíram em cada tipo de decisão.

Essa visão é mais coerente com o estudo das lógicas do que apenas procurar uma métrica única de desempenho.

In [ ]:
tabelas_resumo = []

for nome, resultado in resultados.items():
    resumo = resumo_simples(resultado)
    resumo["experimento"] = nome
    tabelas_resumo.append(resumo)

resumo_geral = pd.concat(tabelas_resumo, ignore_index=True)
resumo_geral = resumo_geral[["experimento", "decisao", "quantidade", "percentual"]]

display(resumo_geral)

## Etapa 8 — Visualizar comparativamente

Cada gráfico mostra a distribuição das decisões de uma lógica em um cenário.

O objetivo é observar o comportamento:

- a multivalorada cria estados intermediários;
- a fuzzy mantém graus e suspeitas graduais;
- a paracompleta aumenta inconclusões quando faltam dados;
- a paraconsistente sinaliza inconsistências quando há conflito.

In [ ]:
for nome, resultado in resultados.items():
    mostrar_distribuicao_decisoes(
        resultado,
        titulo=nome
    )

## Etapa 9 — Construir uma tabela interpretativa

Agora vamos criar uma tabela mais conceitual, ligando cada lógica ao tipo de problema que ela trata melhor.

In [ ]:
tabela_interpretativa = pd.DataFrame([
    {
        "Lógica": "Multivalorada",
        "O que representa melhor": "Estados discretos além de verdadeiro/falso",
        "Exemplo de saída": "baixo, moderado, alto, indeterminado",
        "Quando é mais útil": "Quando a decisão precisa de categorias intermediárias"
    },
    {
        "Lógica": "Fuzzy",
        "O que representa melhor": "Graus de pertinência",
        "Exemplo de saída": "baixa, moderada ou alta suspeita",
        "Quando é mais útil": "Quando a evidência é gradual, não rígida"
    },
    {
        "Lógica": "Paracompleta",
        "O que representa melhor": "Lacunas de informação",
        "Exemplo de saída": "inconclusivo",
        "Quando é mais útil": "Quando ausência de dado não deve virar evidência negativa"
    },
    {
        "Lógica": "Paraconsistente",
        "O que representa melhor": "Conflitos informacionais",
        "Exemplo de saída": "inconsistente",
        "Quando é mais útil": "Quando fontes diferentes podem se contradizer"
    },
])

display(tabela_interpretativa)

## Etapa 10 — Exportar resultados

Os arquivos CSV gerados podem ser baixados pelo painel lateral do Colab.

Eles servem para consultar os resultados completos de cada lógica.

In [ ]:
resumo_geral.to_csv("comparacao_didatica_logicas.csv", index=False)
tabela_interpretativa.to_csv("tabela_interpretativa_logicas.csv", index=False)

for nome, resultado in resultados.items():
    nome_limpo = (
        nome.lower()
        .replace(" — ", "_")
        .replace(" ", "_")
        .replace("ç", "c")
        .replace("ã", "a")
        .replace("á", "a")
        .replace("ó", "o")
        .replace("í", "i")
    )
    resultado.to_csv(f"resultado_{nome_limpo}.csv", index=False)

print("Arquivos exportados com sucesso.")